<a href="https://colab.research.google.com/github/elfantasies/AI/blob/main/0702_Colab_LINE_Bot_with_GEMINI_Stateful.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

In [12]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [13]:
import os
from pyngrok import ngrok

In [14]:
ngrok.kill()

In [15]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://sequential-dispensational-tripp.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://sequential-dispensational-tripp.ngrok-free.dev


True

In [16]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        response_modalities=["TEXT"],
    )
)

In [17]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [18]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Ming Hsin University of Science and Technology），簡稱明新科大，位於臺灣新竹縣，是一所以工學、管理、設計、服務產業為發展重點的科技大學。

以下是明新科技大學的簡要介紹：

1.  **創校歷史與發展：**
    *   創立於1966年，前身為「明新工業專科學校」。
    *   1997年改制為「明新技術學院」。
    *   2002年獲教育部核准升格為「明新科技大學」，逐步發展成為一所綜合型科技大學。

2.  **地理位置與特色：**
    *   學校地理位置優越，鄰近新竹科學園區，與周邊高科技產業及服務業保持密切的產學合作關係。
    *   這使得學校在課程設計、實習機會和就業輔導方面，能緊密結合產業脈動，為學生提供豐富的實務學習與發展機會。

3.  **教育理念與目標：**
    *   明新科大秉持「明德、新民、致知」的校訓，致力於提供實務導向的專業教育。
    *   學校強調理論與實作並重，旨在培養學生具備實際操作能力、解決問題的技能以及創新思維。
    *   其教育目標是培育符合國家及產業發展需求的高素質專業人才，期許學生畢業後能立即投入職場，貢獻所學。

4.  **學院與系所：**
    目前設有四大專業學院：
    *   **工學院：** 涵蓋機械工程、電機工程、資訊工程、土木工程等相關領域，是學校的傳統強項。
    *   **管理學院：** 提供企業管理、行銷與流通管理、資訊管理等學位。
    *   **服務事業學院：** 包含觀光事業、餐飲管理、旅館事業、幼兒保育等，回應現代社會對服務產業人才的需求。
    *   **設計學院：** 設有多媒體與遊戲設計、時尚造型設計等科系，強調創意與實作。

5.  **教學特色：**
    *   **產學合作：** 與眾多企業簽訂合作協議，提供學生實習機會，並共同研發技術。
    *   **實務教學：** 重視實驗、實習與專題製作，讓學生在畢業前即累積實務經驗。
    *   **證照輔導：** 鼓勵學生考取相關專業證照，提升職場競爭力。
    *   **就業導向：** 因應產業需求調整課程內容，畢業生因其紮實的專業技能與實作經驗，深受業界肯定，就業率表現良好。

總體而言，明新科技

In [19]:
result2 = stateful_query("校長是誰？")
print(result2)

截至我所知的最新資訊（通常是2023年末至2024年初），明新科技大學的校長是 **劉國偉** 教授。

劉國偉校長在工程領域具有深厚的學術背景和豐富的行政管理經驗，帶領明新科大在教學、研究及產學合作方面持續發展。


In [20]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [08/Jan/2026 04:25:46] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"Uacb01c8222141eb15a6c0b7aad9e4791","events":[]}


INFO:werkzeug:127.0.0.1 - - [08/Jan/2026 04:35:03] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"Uacb01c8222141eb15a6c0b7aad9e4791","events":[]}
